In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
import re

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization, 
    GlobalAveragePooling1D, MultiHeadAttention, Add, GaussianNoise
)
from tensorflow.keras.regularizers import l2

from sentence_transformers import SentenceTransformer

from sklearn.preprocessing import OneHotEncoder 

warnings.filterwarnings("ignore")

2025-12-14 10:13:39.580385: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765707219.772794      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765707219.827609      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', 'url', text) 
    text = re.sub(r'\s+', ' ', text)  
    text = text.strip()
    return text

# CONFIGURATION

In [3]:
MODEL_PATH = "/kaggle/input/weights/keras/default/1/best_model_NLP.keras"  

TEST_DATA_PATH = "/kaggle/input/test-razan/Test cases 1.xlsx"

OUTPUT_PATH = "submission.csv"

In [4]:

EMBEDDING_DIM = 768  
NUM_CLASSES = 5      
NUM_HEADS = 8
FF_DIM = 256
DROPOUT_RATE = 0.2
L2_REG = 1e-4

In [12]:
test_df = pd.read_excel(TEST_DATA_PATH)
test_df.head()

,id,text
0,1,Best corned beef sandwich I've had anywhere at...
1,2,If Cowboy Ciao is the best restaurant in Scott...
2,3,Wow! Went on a Sunday around 11am - busy but ...
3,4,I have never been here before so I didn't know...
4,5,"Hipster,Trendy ????-I think NOT !!!! Very disa..."


In [14]:

def main():
    print("=" * 80)
    print("SENTIMENT CLASSIFICATION - KAGGLE TEST SCRIPT")
    print("=" * 80)
    
    print("read test data ")
    test_df = pd.read_csv(TEST_DATA_PATH) 
 
    
   
    test_df['text'] = test_df['text'].apply(clean_text)
    
    test_id = test_df['id'].values
    test_texts = test_df['text'].values
    
    
    embedding_model = SentenceTransformer('all-mpnet-base-v2')
    

    X_test_emb = embedding_model.encode(
        test_texts, 
        show_progress_bar=True,
        batch_size=32,
        convert_to_numpy=True
    )
    
    
    model = load_model(
        MODEL_PATH,
        custom_objects={
            'MultiHeadAttention': MultiHeadAttention,
            'GaussianNoise': GaussianNoise
        }
    )
    predictions = model.predict(X_test_emb, batch_size=32, verbose=1)
    predicted_indices = np.argmax(predictions, axis=1)
  
    known_classes = ['Bad', 'Excellent', 'Good', 'Very bad', 'Very good']
    
    ohe = OneHotEncoder(sparse_output=False)
    ohe.fit(np.array(known_classes).reshape(-1, 1))
   
    predicted_labels = ohe.categories_[0][predicted_indices]
    submission = pd.DataFrame({
        'id': test_id,
        'review': predicted_labels
    })
    
    submission.to_csv(OUTPUT_PATH, index=False)
    print("Top 10 Predictions:")
    print(submission.head(10))

    if __name__ == "__main__":
        main()

SENTIMENT CLASSIFICATION - KAGGLE TEST SCRIPT
read test data 
Loaded 6 test samples
Columns: ['id', 'text']
sentence embedding Load


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

I0000 00:00:1765708449.831638      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14941 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


TESTING


I0000 00:00:1765708451.258995     174 service.cc:148] XLA service 0x7f9768006570 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1765708451.259727     174 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1765708451.345758     174 cuda_dnn.cc:529] Loaded cuDNN version 90300


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step
Generated predictions with shape: (6, 5)
Creating submission csv 
   id     review
0   1  Excellent
1   2        Bad
2   3  Excellent
3   4       Good
4   5        Bad
5   6  Excellent


I0000 00:00:1765708451.577962     174 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
